# CUDA SGEMM Optimization - run on Colab / Kaggle
1. **Runtime → Change runtime type → T4 GPU** (Kaggle: Settings → Accelerator → GPU T4).
2. Run the cells top to bottom. Results land in `results/`.


In [ ]:
!nvidia-smi
!nvcc --version | tail -2
!which ncu || echo 'ncu not on PATH'

## Get the code
Option A: clone your GitHub repo (edit the URL). Option B: upload the project zip.

In [ ]:
REPO_URL = "https://github.com/<your-username>/cuda-gemm-optimization.git"  # <- edit
import os
if not os.path.exists('cuda-gemm-optimization'):
    if '<your-username>' not in REPO_URL:
        !git clone $REPO_URL
    else:
        from google.colab import files  # Colab only; on Kaggle add the zip as a dataset
        up = files.upload()             # choose cuda-gemm-optimization.zip
        !unzip -q -o cuda-gemm-optimization.zip
%cd cuda-gemm-optimization
!git pull 2>/dev/null || true

## Build
`ARCH=sm_75` is the T4. On Kaggle P100 use `ARCH=sm_60` (no tensor cores: run without kernels 8/9).

In [ ]:
!test -f Makefile || cp build.mk Makefile   # Makefile ships as build.mk in some copies
!make ARCH=sm_75
!grep -E 'Function properties|registers' results_ptxas.txt | head -40

## Optional: CPU-emulator self-check (no GPU needed)

In [ ]:
!make test-emu | tail -25

## Run all kernels
K1 at 4096 takes a few seconds per launch; total runtime is a few minutes.

In [ ]:
!./gemm --kernel all --sizes 128 256 512 1024 2048 4096 --reps 10 --csv results/results.csv

## Plots

In [ ]:
!python scripts/plot_perf.py results/results.csv
from IPython.display import Image, display
display(Image('results/perf_vs_size.png')); display(Image('results/pct_cublas.png'))

## Nsight Compute profiling
If you get `ERR_NVGPUCTRPERM`, counters are blocked on this platform - try Kaggle, or skip this step.

In [ ]:
!bash scripts/profile_all.sh

## Roofline

In [ ]:
!python scripts/roofline.py --results results/results.csv --size 4096
display(Image('results/roofline.png'))

## Save results
Download the zip, or `git add results && git commit && git push` if you cloned with credentials.

In [ ]:
!zip -q -r results.zip results results_ptxas.txt
try:
    from google.colab import files; files.download('results.zip')
except ImportError:
    print('On Kaggle: results.zip is in /kaggle/working - download it from the Output panel')